# 🌲 Random Forests
**ISLP Chapter 8 · Pattern Recognition for the Rest of Us**

> One of the most powerful and widely-used ML models. Random Forests fix the main weakness of individual decision trees — high variance — by averaging hundreds of trees, each trained on a random subset of data and features.

### What you'll learn
- How bagging reduces variance through averaging
- The random feature selection trick that decorrelates trees
- Out-of-bag (OOB) error — a free estimate of test error
- Variable importance — which features matter most
- Tuning: n_estimators, max_features, max_depth

### Dataset: Heart disease prediction (Cleveland Heart Disease dataset)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.inspection import permutation_importance

In [ ]:
!pip install ISLP -q
from ISLP import load_data
heart = load_data('Heart').dropna()
heart = pd.get_dummies(heart, drop_first=True, dtype=float)
X = heart.drop('AHD_Yes', axis=1) if 'AHD_Yes' in heart.columns else heart.drop(heart.columns[-1], axis=1)
y = (heart['AHD_Yes'] if 'AHD_Yes' in heart.columns else heart.iloc[:,-1]).astype(int)
print(f"Heart: {X.shape}  |  Disease rate: {y.mean():.1%}")

## 🌳 Part 1 — From Decision Trees to Random Forests

A single **decision tree** splits data by asking questions (Is age > 55? Is cholesterol > 240?) until it reaches pure leaf nodes. Trees are intuitive but have **high variance** — small changes in data produce very different trees.

**Bagging** (Bootstrap Aggregating): Train B trees on B bootstrap samples, average their predictions. Reduces variance by ~1/B (if trees were independent).

**Random Forests** add one key twist: at each split, only consider a random subset of **m** features (typically m=√p for classification). This **decorrelates** the trees — even with the same data, trees will differ. Decorrelated trees average much better than correlated ones.

In [ ]:
# Show: single tree vs random forest
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Single deep tree — high variance
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_tr, y_tr)

# Random Forest — average of 500 trees
rf = RandomForestClassifier(n_estimators=500, oob_score=True, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

print("=== Single Decision Tree vs Random Forest ===\n")
print(f"{'Metric':<25} {'Decision Tree':>15} {'Random Forest':>15}")
print("-"*55)
print(f"{'Training accuracy':<25} {tree.score(X_tr,y_tr):>15.3f} {rf.score(X_tr,y_tr):>15.3f}")
print(f"{'Test accuracy':<25} {tree.score(X_te,y_te):>15.3f} {rf.score(X_te,y_te):>15.3f}")
print(f"{'OOB error':<25} {'N/A':>15} {1-rf.oob_score_:>15.3f}")
print(f"{'Generalization gap':<25} {tree.score(X_tr,y_tr)-tree.score(X_te,y_te):>15.3f} {rf.score(X_tr,y_tr)-rf.score(X_te,y_te):>15.3f}")
print("\n📌 Decision tree fits training data perfectly (100%) but overfits — high generalization gap")
print("   Random forest has a much smaller gap — it generalizes far better")

In [ ]:
# How test error changes with number of trees
n_trees_range = [1,5,10,25,50,100,200,500]
oob_errors, test_errors = [], []

for n in n_trees_range:
    rf_n = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1)
    rf_n.fit(X_tr, y_tr)
    oob_errors.append(1 - rf_n.oob_score_)
    test_errors.append(1 - rf_n.score(X_te, y_te))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n_trees_range, test_errors, 'o-', color='#1e5fa8', lw=2, label='Test error', markersize=6)
ax.plot(n_trees_range, oob_errors,  's--', color='#e85d20', lw=2, label='OOB error', markersize=6)
ax.set_xlabel('Number of Trees')
ax.set_ylabel('Error Rate')
ax.set_title('Error vs Number of Trees — error stabilizes around 100-200 trees')
ax.legend()
ax.set_xscale('log')
plt.tight_layout()
plt.show()
print("\n📌 OOB error ≈ test error — it's a free cross-validation estimate!")
print("   Error stops improving much beyond ~100-200 trees.")

In [ ]:
# Variable Importance — which features drive predictions?
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#e85d20' if v > importances.quantile(0.75) else '#1e5fa8' for v in importances]
ax.barh(importances.index, importances.values, color=colors, edgecolor='white')
ax.set_xlabel('Mean Decrease in Gini Impurity')
ax.set_title('Random Forest Variable Importance\n(orange = top quartile)')
plt.tight_layout()
plt.show()

top3 = importances.tail(3).index.tolist()
print(f"\n📌 Top 3 predictors of heart disease: {', '.join(reversed(top3))}")
print("   These features appear most often at the top of trees and give the biggest purity gains.")

In [ ]:
# Hyperparameter tuning: max_features
max_feat_options = ['sqrt', 'log2', 1, 3, 5, 8, X.shape[1]]
results = []
for mf in max_feat_options:
    rf_mf = RandomForestClassifier(n_estimators=300, max_features=mf, oob_score=True,
                                    random_state=42, n_jobs=-1)
    rf_mf.fit(X_tr, y_tr)
    results.append({'max_features': str(mf), 'test_acc': rf_mf.score(X_te,y_te), 
                    'oob_acc': rf_mf.oob_score_})

results_df = pd.DataFrame(results)
print("=== Effect of max_features ===")
print(results_df.to_string(index=False))
print("\n📌 'sqrt' (√p features per split) is usually the best default for classification.")

In [ ]:
answers = {
    "q1": "",  # What is the key difference between bagging and Random Forests?
    "q2": "",  # Why does OOB error give a valid estimate of test error?
    "q3": "",  # What does variable importance measure in a Random Forest?
    "q4": "",  # Why does adding more trees eventually stop helping?
    "q5": "",  # When might a single decision tree be preferred over Random Forest?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*